# ✈️ Extracción y Procesamiento Inicial de Datos de Tráfico Aéreo (OpenSky)

La disponibilidad de datos aeronáuticos en tiempo real permite analizar patrones de movilidad aérea, comportamiento de aeronaves y variaciones operativas a escala global.  
Este proyecto desarrolla un **pipeline ETL** que ingesta información del endpoint público de **OpenSky Network**, centrado en capturar el estado actual de miles de vuelos activos en simultáneo.

La API proporciona variables clave como:

- identificador **ICAO24**  
- país de origen  
- latitud / longitud  
- altitudes barométrica y geométrica  
- velocidad, rumbo, tasa vertical  
- estado en tierra o en vuelo  
- timestamp del servidor  

Estos datos se transforman y almacenan en un **Data Lake local** siguiendo la arquitectura **Bronze → Silver → Gold**, lo que permite realizar análisis históricos, construir métricas aeronáuticas y preparar la futura migración a entornos de nube (Azure).

## Objetivos

**Extracción (Bronze):**
- Consumir el endpoint `states/all` de OpenSky.  
- Normalizar la estructura JSON y convertirla en tabla.  
- Incorporar timestamps (servidor y extracción).  
- Guardar los datos crudos en **Delta Lake**.

**Transformación (Silver):**
- Limpiar valores faltantes y tipos de datos.  
- Estandarizar columnas y coordenadas.  
- Preparar la tabla para análisis temporal.

**Métricas (Gold):**
- Crear features de movilidad aérea: altitud efectiva, variación de velocidad, indicadores de vuelo/estacionamiento, etc.  
- Generar datasets optimizados para visualización y análisis exploratorio.

## Alcance y supuestos

- Se utilizan exclusivamente datos públicos provistos por OpenSky Network.  
- La extracción se realiza bajo límites de la API pública (sin autenticación obligatoria).  
- El objetivo es **práctico y educativo**, orientado al portfolio de Ingeniería de Datos.

## Reproducibilidad

- Dependencias detalladas en `requirements.txt`.  
- Las rutas del Data Lake se configuran en `pipeline.conf`.  
- Todas las funciones auxiliares se encuentran en `src/etl_utils.py`.

---

**Estructura del notebook:**

0) Configuración inicial  
1) Extracción del endpoint `states/all`  
2) Normalización del JSON  
3) Limpieza y estandarización mínima  
4) Almacenamiento en Delta Lake (capa Bronze)  
5) Verificación y vista preliminar de los datos  

## 0. Configuración inicial

En este paso se importan todas las librerías necesarias y las funciones auxiliares definidas en `etl_utils.py`.  
Este enfoque permite mantener el notebook **ordenado, modular y fácilmente reproducible**, centralizando en un único módulo las operaciones comunes del pipeline ETL: extracción desde la API pública de **OpenSky Network**, normalización del JSON, estandarización de columnas y escritura en las distintas capas del **Data Lake local** (Bronze → Silver → Gold).

El objetivo de esta sección es garantizar que todas las dependencias estén correctamente cargadas antes de iniciar el proceso de extracción y almacenamiento.

In [6]:
import sys
import os

# Se agrega la carpeta src al path (sube un nivel desde /notebooks)
sys.path.append(os.path.abspath("../src"))

# Importar funciones auxiliares
from etl_utils import *

# Librerías comunes
import pandas as pd

print("✅ Librerías importadas correctamente.")

✅ Librerías importadas correctamente.


## 1. Autenticación y lectura de configuración

La configuración del proyecto se administra mediante el archivo `pipeline.conf`,  
que centraliza parámetros como:
- la **URL base** de la API de OpenSky Network  
- credenciales opcionales para *Basic Auth* (en caso de usarse)  
- rutas del **Data Lake local**

Aunque la API pública de OpenSky no requiere autenticación obligatoria,almacenar parámetros en un archivo de configuración permite:
- mantener el notebook limpio  
- evitar credenciales expuestas en el código  
- facilitar la migración futura a servicios en la nube (Azure Key Vault)

El archivo se lee mediante `ConfigParser`, lo que permite obtener los valores  
en forma segura y reusable.


In [7]:
# Se instancia el parser y se lee el archivo de configuración
from configparser import ConfigParser

parser = ConfigParser()
parser.read("../pipeline.conf")

['../pipeline.conf']

In [8]:
# Parámetros de conexión
api_config = parser["api-opensky"]
base_url = api_config["base_url"]

In [9]:
print("📄 Configuración cargada correctamente.")
print(f"URL base: {base_url}")

📄 Configuración cargada correctamente.
URL base: https://opensky-network.org/api/states/all


In [10]:
# Prueba de conexión a la API OpenSky
response = requests.get(base_url)

if response.status_code == 200:
    data = response.json()
    print(f"La petición fue exitosa. Tipo de respuesta: {type(data)}")

    # Claves principales del JSON
    print(f"Claves principales recibidas: {list(data.keys())[:5]}")

    # Inspección parcial de 'states'
    print("\nPrimeras 2 aeronaves registradas:")
    pprint(data["states"][:2])

else:
    print(f"❌ Error en la petición: {response.status_code}, {response.content}")
print("✅ Prueba de conexión a la API realizada.")

La petición fue exitosa. Tipo de respuesta: <class 'dict'>
Claves principales recibidas: ['time', 'states']

Primeras 2 aeronaves registradas:
[['a89ea5',
  'N6545H  ',
  'United States',
  1763129391,
  1763129391,
  -102.5469,
  32.4474,
  1104.9,
  False,
  57.53,
  164.97,
  0,
  None,
  1127.76,
  None,
  False,
  0],
 ['3ffc29',
  'DMFSS   ',
  'Germany',
  1763129351,
  1763129351,
  12.3874,
  48.2546,
  647.7,
  False,
  44.8,
  357.37,
  -0.33,
  None,
  None,
  None,
  False,
  0]]
✅ Prueba de conexión a la API realizada.


## 2. Capa Bronze — Extracción y almacenamiento de datos crudos

En esta etapa se realiza la **extracción directa de datos** desde la API pública de **OpenSky Network**, que provee información en tiempo real sobre aeronaves detectadas a nivel global.  
El objetivo es obtener el conjunto completo de registros tal como es devuelto por el endpoint `states/all` y conservarlo en su forma más fiel dentro de la capa **🟤 Bronze** del Data Lake.

Los datos obtenidos incluyen:

- Identificador único **ICAO24**  
- Indicativo de llamada (**callsign**)  
- País de origen  
- Posición geográfica (latitud, longitud)  
- Altitudes barométrica y geométrica  
- Velocidad, rumbo y tasa vertical  
- Estado operativo (en tierra o en vuelo)  
- Timestamp del servidor (`time`) correspondiente a la captura

Cada aeronave es representada inicialmente como una lista ordenada de 17 elementos, por lo que en esta etapa se prioriza **preservar los datos crudos** antes de aplicar procesos de estructuración o limpieza.

Los datos se almacenan en **formato Delta Lake**, dentro del directorio:

`data/etl_datalake/bronze/api_opensky/`

empleando el modo **`overwrite`**, ya que la información corresponde a un snapshot puntual del estado global del tráfico aéreo y puede reemplazarse completamente en cada actualización.


### 2.1 Extracción de datos estáticos

Se realiza una *ingesta full* sobre el recurso estático `aircraftDatabase.csv` provisto por OpenSky Network, que contiene información descriptiva sobre aeronaves (modelo, fabricante, typecode, operador, entre otros metadatos relevantes).

Dado que este archivo cambia muy poco en el tiempo, se descarga en su totalidad en cada ejecución y se sobrescribe la versión previa (mode="overwrite").  
Este enfoque evita duplicados, simplifica el pipeline y garantiza la disponibilidad de la versión más reciente.

Los datos se almacenan en la capa 🟤 *Bronze* en formato *Delta Lake*.

In [13]:
# --- Extracción de datos estáticos — OpenSky aircraft metadata ---

# URL del recurso estático oficial
metadata_url = "https://opensky-network.org/datasets/metadata/aircraftDatabase.csv"

# Descarga del archivo CSV
# Nota: Estos metadatos cambian muy poco, por lo que se aplicará ingesta "full"
df_aircraft = pd.read_csv(metadata_url)

# Vista preliminar del dataset cargado
print("📄 Dataset de metadatos cargado correctamente.")
print(f"Filas: {df_aircraft.shape[0]} | Columnas:, {df_aircraft.shape[1]}")
df_aircraft.head(3)

📄 Dataset de metadatos cargado correctamente.
Filas: 520000 | Columnas:, 27


,icao24,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,...,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categoryDescription
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
1,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,NaN,L1P,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
2,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,NaN,L2P,NaN,...,NaN,1977-01-01,NaN,NaN,LYCOMING TI0-540 SER,False,False,False,NaN,NaN


### 2.2 Extracción de datos dinámicos

Para los datos dinámicos se utiliza la información proveniente del endpoint `states/all`, que ofrece el estado en tiempo real de miles de aeronaves a nivel global.  
Dado que estos datos se actualizan continuamente, cada ejecución captura un *snapshot* independiente del tráfico aéreo del momento.

En este caso no se aplica una actualización incremental, ya que el conjunto de aeronaves presentes varía en cada consulta y no existe un identificador temporal que permita rastrear cambios de manera estricta.  
Por ello, cada snapshot se almacena íntegramente en la capa 🟤 *Bronze*, preservando la evolución temporal entre ejecuciones y habilitando análisis posteriores en Silver y Gold.

In [17]:
# --- Extracción de datos dinámicos — OpenSky states/all ---

# Obtención del snapshot dinámico mediante la función auxiliar
json_data = get_opensky_states()

# Vista preliminar del JSON recibido
print("🔑 Claves principales:", list(json_data.keys()))
print("🛫 Cantidad de aeronaves detectadas:", len(json_data["states"]))

# Visualización de los primeros registros crudos
print("\nPrimeras 2 aeronaves (formato crudo):")
pprint(json_data["states"][:2])

🔑 Claves principales: ['time', 'states']
🛫 Cantidad de aeronaves detectadas: 11665

Primeras 2 aeronaves (formato crudo):
[['39de4e',
  'TVF73VH ',
  'France',
  1763131835,
  1763131835,
  0.0269,
  48.2694,
  10797.54,
  False,
  203.91,
  235.59,
  5.85,
  None,
  10873.74,
  '7644',
  False,
  0],
 ['a89ea5',
  'N6545H  ',
  'United States',
  1763131675,
  1763131675,
  -102.4832,
  32.4335,
  1104.9,
  False,
  55.68,
  176.29,
  -0.65,
  None,
  1127.76,
  None,
  False,
  0]]


### 2.3 Guardado en Bronze — Delta Lake

Una vez realizada la extracción de los datos estáticos y dinámicos, ambos recursos se almacenan en la capa 🟤 *Bronze* del Data Lake en formato **Delta Lake**.  
En esta etapa no se aplican transformaciones ni procesos de limpieza: los datos se preservan tal como fueron obtenidos desde la API, cumpliendo el propósito de Bronze como zona de almacenamiento crudo.

Los metadatos estáticos se guardan mediante una *ingesta full* (mode="overwrite"), ya que su contenido cambia muy poco y resulta más simple reemplazar el dataset completo en cada ejecución.

Los datos dinámicos provenientes de `states/all` se guardan como snapshots independientes, manteniendo la trazabilidad temporal de cada captura.  
Este enfoque permite conservar el historial de estados del tráfico aéreo para posteriores análisis en las capas Silver y Gold.

Las rutas de salida se definen dentro del directorio:

`data/etl_datalake/bronze/`

utilizando subdirectorios separados para **datos estáticos** y **datos dinámicos**.

In [22]:
# --- Definición de rutas del Data Lake (Bronze) ---

# Se mantiene fuera de la carpeta notebooks para centralizar los datos.
datalake_root = "../data/etl_datalake"

# Carpeta raíz del dominio OpenSky dentro de Bronze
bronze_dir = f"{datalake_root}/bronze/api_opensky"

# Subcarpetas descriptivas según el tipo de recurso
static_dir  = f"{bronze_dir}/aircraft_metadata"
dynamic_dir = f"{bronze_dir}/states"

print("📁 Rutas definidas:")
print("Static  →", static_dir)
print("Dynamic →", dynamic_dir)

📁 Rutas definidas:
Static  → ../data/etl_datalake/bronze/api_opensky/aircraft_metadata
Dynamic → ../data/etl_datalake/bronze/api_opensky/states


In [23]:
# --- Guardado en Bronze — Metadatos estáticos (Delta Lake) ---

# Ingesta full: se sobrescribe el dataset completo en cada ejecución,
# dado que los metadatos de aeronaves cambian muy poco en el tiempo.
save_data_as_delta(
    df=df_aircraft,
    path=static_dir,
    mode="overwrite"
)

print("🟤 Datos estáticos guardados correctamente en Bronze (Delta Lake).")

💾 Datos guardados en Delta Lake: ../data/etl_datalake/bronze/api_opensky/aircraft_metadata
🟤 Datos estáticos guardados correctamente en Bronze (Delta Lake).


In [24]:
# --- Guardado en Bronze — Snapshot dinámico (Delta Lake) ---

# Conversión a DataFrame del recurso dinámico "states"
df_dynamic = pd.DataFrame(json_data["states"])

# En algunos snapshots, ciertas columnas pueden venir completamente vacías.
# Delta Lake no acepta columnas de tipo "Null" (100% None), por lo que se eliminan
# únicamente aquellas que no contienen ningún valor real.
df_dynamic = df_dynamic.dropna(axis=1, how="all")

print("📄 Shape del snapshot después de eliminar columnas vacías:", df_dynamic.shape)

# Guardado incremental (append) para preservar el historial de snapshots.
save_data_as_delta(
    df=df_dynamic,
    path=dynamic_dir,
    mode="append"
)

print("🟤 Snapshot dinámico guardado correctamente en Bronze (Delta Lake).")

📄 Shape del snapshot después de eliminar columnas vacías: (11665, 16)
💾 Datos guardados en Delta Lake: ../data/etl_datalake/bronze/api_opensky/states
🟤 Snapshot dinámico guardado correctamente en Bronze (Delta Lake).


## 3. Capa Silver — Normalización y limpieza

La capa **Silver** aplica transformaciones sobre los datos almacenados en Bronze con el objetivo de obtener un conjunto de datos limpio, tabular y estructurado, adecuado para análisis posteriores.

En esta etapa se realizan:
- renombrado de columnas  
- estandarización de tipos  
- conversión de timestamps  
- selección de atributos relevantes  
- reducción de nulos  
- enriquecimiento opcional mediante cruce entre datasets

El procesamiento se organiza en dos pasos:

### 3.1 Metadatos estáticos (aircraft_metadata)
Depuración de columnas, estandarización de nombres y preparación del diccionario de aeronaves.

### 3.2 Datos dinámicos (states/all)
Normalización del snapshot, asignación de nombres descriptivos a las columnas, tipificación y preparación para análisis temporal.

Este enfoque garantiza un esquema consistente y una base sólida para la etapa **Gold**, donde se generarán métricas, indicadores y visualizaciones del tráfico aéreo.

### 3.1 Metadatos estáticos — Normalización y selección de atributos

El dataset de metadatos de aeronaves, obtenido desde el endpoint de OpenSky, contiene información descriptiva asociada a cada código `icao24`.  
Si bien estos datos cambian muy poco en el tiempo, llegan en un formato heterogéneo, con numerosas columnas incompletas o totalmente vacías.

En esta etapa se realiza una depuración orientada a construir un diccionario de aeronaves confiable y estable para su uso posterior:

- **estandarización de nombres de columnas**  
- **eliminación de atributos irrelevantes o con nulos del 100%**  
- **selección de campos útiles** como:  
  - `icao24`  
  - `registration`  
  - `manufacturername`  
  - `model`  
  - `typecode`  
  - `operator`  
- **verificación de duplicados** por clave primaria `icao24`  
- **tipificación suave** (strings y categóricos)

El objetivo es obtener una tabla limpia y coherente que servirá como fuente de enriquecimiento para los datos dinámicos procesados en la siguiente etapa.